# Сравнение ЦПТ-фильтров для наблюдений Парето с точным фильтром

На одной скрытой траектории $(\theta, Y)$ и одном потоке наблюдений Парето с шагом $h_t=1$ сравниваются:

- **точный фильтр Парето:** исходная плотность наблюдения $X_k=Y_{1,k}+Y_{2,k}\varepsilon_k$ на сетке с шагом $h_t=1$;
- **ЦПТ-фильтр при $h=1$:** гауссовское приближение $X_k\mid Y_k\approx\mathcal N(Y_{1,k}+Y_{2,k},\,Y_{2,k}^2/12)$ для каждого исходного наблюдения;
- **ЦПТ-фильтры для блоков:** гауссовское приближение для сумм непересекающихся блоков длины $n$ из архивного набора размеров.

Архив `saved_path_clt_averaging/comparison.npz`, созданный `scripts/run_clt_averaging.py`, содержит точный фильтр и все ЦПТ-фильтры, включая гауссовский прогон при $h=1$. Ноутбук только загружает архивные массивы и строит по ним диагностику.

In [ ]:
import _bootstrap  # noqa: F401

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 12,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

ROOT = Path.cwd().resolve()
if not (ROOT / 'saved_path_clt_averaging' / 'comparison.npz').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'saved_path_clt_averaging'
assert (DATA_DIR / 'comparison.npz').exists(), 'Запустите ноутбук из корня репозитория или notebooks/'

with np.load(DATA_DIR / 'comparison.npz', allow_pickle=False) as archive:
    data = {name: archive[name] for name in archive.files}

theta, y, jump_times = data['theta'], data['y'], data['t']
obs_matrix = data['obs']
obs = obs_matrix[:, 0]
ns = [int(n) for n in data['ns']]
rerun_message = ('Архив не содержит полный прогон h=1. '
                 'Перезапустите scripts/run_clt_averaging.py --ns 1,10,20,50.')
if 1 not in ns:
    raise RuntimeError(rerun_message)
required_by_n = ('obs', 'theta_est_clt', 'theta_est_delta_clt', 'y_est_clt', 't_net')
missing = [f'{prefix}_n{n}' for n in ns for prefix in required_by_n
           if f'{prefix}_n{n}' not in data]
if missing:
    raise RuntimeError(f'{rerun_message} Нет ключей: {", ".join(missing)}')
n_exact = int(data['n_exact'])
ht_exact = float(data['ht_exact'])
t_exact = data['t_net_exact']
th_exact = data['theta_est_exact']
y_exact = data['y_est_exact']

n_colors = cm.viridis(np.linspace(0.15, 0.9, len(ns)))
n_color = dict(zip(ns, n_colors))


def clt_label(n):
    return 'ЦПТ (Gaussian, h=1)' if n == 1 else f'ЦПТ, сумма блока n={n}'


print(f'Размеры блоков n = {ns}')
print(f'Скрытая траектория: {len(theta)} состояний/сегментов; '
      f'{n_exact} исходных наблюдений (h_t={ht_exact:g}); '
      f'горизонт {t_exact[-1] / 3600:.3f} ч.')
print(f'Оценки точного фильтра: {th_exact.shape}')
for n in ns:
    th_n = data[f'theta_est_clt_n{n}']
    print(f'  n={n:>4}: {th_n.shape[0]} точек на сетке фильтра '
          f'(t_net_n{n}[-1]={data[f"t_net_n{n}"][-1]:.0f})')

## Проверки целостности

Для каждого $n$ архивные наблюдения должны совпадать с суммами непересекающихся блоков исходного потока, а сетки времени и массивы оценок должны иметь согласованные размеры. Проверки $h=1$ используют только загруженные массивы и размеры архивных точных оценок.

In [ ]:
print('Проверки целостности')
for n in ns:
    obs_n = data[f'obs_n{n}'][:, 0]
    expected = obs[:n_exact].reshape(-1, n).sum(axis=1)
    assert obs_n.shape == expected.shape
    assert np.allclose(obs_n, expected)

    t_net_n = data[f't_net_n{n}']
    th_n = data[f'theta_est_clt_n{n}']
    y_n = data[f'y_est_clt_n{n}']
    n_blocks = n_exact // n
    assert t_net_n.shape == (n_blocks + 1,)
    assert th_n.shape == (n_blocks + 1, th_exact.shape[1])
    assert y_n.shape == (n_blocks + 1, y_exact.shape[1])
    print(f'  n={n:>4}: наблюдения и суммы блоков совпадают; '
          f'{n_blocks} блоков, сетки и оценки согласованы (OK)')

assert th_exact.shape[0] == t_exact.shape[0] == n_exact + 1
assert y_exact.shape[0] == n_exact + 1

th_n1 = data['theta_est_clt_n1']
th_delta_n1 = data['theta_est_delta_clt_n1']
y_n1 = data['y_est_clt_n1']
assert data['obs_n1'].shape == obs_matrix.shape
assert np.array_equal(data['obs_n1'], obs_matrix)
assert np.array_equal(data['t_net_n1'], t_exact)
assert th_n1.shape == th_delta_n1.shape == (n_exact + 1, th_exact.shape[1])
assert y_n1.shape == (n_exact + 1, y_exact.shape[1])
assert np.isfinite(th_n1).all() and np.isfinite(th_delta_n1).all()
assert np.isfinite(y_n1).all()
assert np.all(th_n1 >= -1e-12) and np.all(th_delta_n1 >= -1e-12)
assert np.allclose(th_n1.sum(axis=1), 1.0, atol=1e-10)
assert np.allclose(th_delta_n1.sum(axis=1), 1.0, atol=1e-10)

print(f'  точный фильтр: {n_exact + 1} точек на сетке h_t={ht_exact:g} (OK)')
print('  ЦПТ h=1: формы, конечность, неотрицательность и нормировка выполнены (OK)')

## Становится ли шум Парето более гауссовским при агрегировании?

Одиночное исходное наблюдение стандартизуется условным средним $Y_1+Y_2$ и условным стандартным отклонением $Y_2/\sqrt{12}$ на истинной траектории. Это одновременно наблюдение для ЦПТ-фильтра при $h=1$, поэтому отдельная дублирующая панель для $n=1$ не строится. Для остальных $n$ условные средние и дисперсии суммируются внутри блока — именно эти два момента использует ЦПТ-правдоподобие. Эталонная кривая — $N(0,1)$.

In [ ]:
def path_at(grid):
    # theta[k], y[k] действуют от предыдущего скачка до jump_times[k].
    idx = np.searchsorted(jump_times, grid, side='left')
    return theta[idx], y[idx]


_, y_step = path_at(t_exact[1:])
mu_step = y_step[:, 0] + y_step[:, 1]
var_step = y_step[:, 1] ** 2 / 12.0
z_step = (obs - mu_step) / np.sqrt(var_step)

# Одиночный шаг уже соответствует n=1; отдельные панели нужны только для блоков n>1.
residual_ns = [n for n in ns if n != 1]
z_by_n = {
    n: (data[f'obs_n{n}'][:, 0] - mu_step.reshape(-1, n).sum(axis=1))
       / np.sqrt(var_step.reshape(-1, n).sum(axis=1))
    for n in residual_ns
}

all_z = np.concatenate([z_step] + [z_by_n[n] for n in residual_ns])
lo = min(-4.0, float(np.floor(np.percentile(all_z, 0.5))))
hi = max(8.0, float(np.ceil(np.percentile(all_z, 99.5))))
x = np.linspace(lo, hi, 800)
normal_pdf = np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)
bins = np.linspace(lo, hi, 121)

fig, axes = plt.subplots(1, 1 + len(residual_ns),
                         figsize=(4.5 * (1 + len(residual_ns)), 4.2),
                         sharey=True, layout='constrained')
axes = np.atleast_1d(axes)
panels = [(axes[0], z_step, 'Одиночное наблюдение Парето ($h_t=1$)', 'tab:red')]
for i, n in enumerate(residual_ns):
    panels.append((axes[i + 1], z_by_n[n], f'Сумма блока, n={n}', n_color[n]))
for ax, values, title, color in panels:
    ax.hist(values, bins=bins, density=True, alpha=.55, color=color, label='эмпирическая')
    ax.plot(x, normal_pdf, color='black', lw=1.6, label='$N(0,1)$')
    ax.set(title=title, xlabel='стандартизованный остаток', xlim=(lo, hi))
    ax.legend(fontsize=10)
axes[0].set_ylabel('плотность')
plt.show()

rows = [('n=1 (один шаг)', z_step)] + [(f'n={n}', z_by_n[n]) for n in residual_ns]
for label, values in rows:
    centered = (values - values.mean()) / values.std()
    skewness = np.mean(centered ** 3)
    excess_kurtosis = np.mean(centered ** 4) - 3
    print(f'{label:16s}: mean={values.mean(): .3f}, std={values.std():.3f}, '
          f'skew={skewness:5.2f}, excess kurtosis={excess_kurtosis:6.1f}, '
          f'P(|Z|>3)={np.mean(np.abs(values) > 3):.3%}, max={values.max():.2f}')

## Оценки вероятностей состояний

Точный фильтр Парето и ЦПТ-фильтр с гауссовским правдоподобием при $h=1$ показаны на одной исходной сетке, но имеют разные явные подписи. Остальные ЦПТ-фильтры показаны на своих сетках блоков. Чёрная ступенчатая линия обозначает истинное скрытое состояние.

In [ ]:
starts = np.r_[0.0, jump_times[:-1]] / 3600
ends = jump_times / 3600
segment_edges = np.r_[starts, ends[-1]]

n_states = th_exact.shape[1]
clt_linestyles = ['-', '--', '-.', ':']
fig, axes = plt.subplots(n_states, 1, figsize=(14, 2.3 * n_states), sharex=True)
axes = np.atleast_1d(axes)
for state, ax in enumerate(axes):
    ax.stairs(theta == state, segment_edges, color='black', lw=2.1, alpha=.9, zorder=2,
              label='истинное состояние')
    ax.plot(t_exact / 3600, th_exact[:, state], color='tab:red', lw=1.6, zorder=4,
            label=f'точный Парето, $h_t={ht_exact:g}$')
    for i, n in enumerate(ns):
        ax.plot(data[f't_net_n{n}'] / 3600, data[f'theta_est_clt_n{n}'][:, state],
                color=n_color[n], ls=clt_linestyles[i % len(clt_linestyles)],
                lw=1.2, zorder=3, label=clt_label(n))
    ax.set(ylabel=fr'$P(\theta={state + 1})$', ylim=(-.03, 1.03))
    ax.grid()
axes[-1].set(xlabel='время, часы', xlim=(0, t_exact[-1] / 3600))
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=len(ns) + 2,
           frameon=False, bbox_to_anchor=(.5, .99))
fig.tight_layout(rect=(0, 0, 1, .94))
plt.show()

## Оценки непрерывных компонент

То же сравнение для $Y_1$ и $Y_2$. Чёрная ступенчатая линия обозначает истинную кусочно-постоянную траекторию; точный фильтр Парето и ЦПТ-фильтр при $h=1$ подписаны отдельно.

In [ ]:
clt_linestyles = ['-', '--', '-.', ':']
fig, axes = plt.subplots(2, 1, figsize=(14, 6.5), sharex=True)
for component, ax in enumerate(axes):
    ax.stairs(y[:, component], segment_edges, color='black', lw=2.1, alpha=.9, zorder=2,
              label='истинное состояние')
    ax.plot(t_exact / 3600, y_exact[:, component], color='tab:red', lw=1.6, zorder=4,
            label=f'точный Парето, $h_t={ht_exact:g}$')
    for i, n in enumerate(ns):
        ax.plot(data[f't_net_n{n}'] / 3600, data[f'y_est_clt_n{n}'][:, component],
                color=n_color[n], ls=clt_linestyles[i % len(clt_linestyles)],
                lw=1.2, zorder=3, label=clt_label(n))
    ax.set_ylabel(fr'$\hat Y_{component + 1}$')
    ax.grid()
axes[-1].set(xlabel='время, часы', xlim=(0, t_exact[-1] / 3600))
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=len(ns) + 2,
           frameon=False, bbox_to_anchor=(.5, .99))
fig.tight_layout(rect=(0, 0, 1, .92))
plt.show()

## Точность относительно истинной траектории

Каждая ЦПТ-оценка, включая $n=1$, сравнивается с истинной траекторией на собственной сетке фильтра. RMSE для $\theta$ рассчитывается по всем четырём one-hot компонентам. RMSE точного фильтра Парето на сетке $h_t=1$ показан горизонтальной линией.

In [ ]:
def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


n_states = th_exact.shape[1]
true_state_exact, true_y_exact = path_at(t_exact)
true_onehot_exact = np.eye(n_states)[true_state_exact]
rmse_exact = {
    'theta': rmse(th_exact, true_onehot_exact),
    'Y1': rmse(y_exact[:, 0], true_y_exact[:, 0]),
    'Y2': rmse(y_exact[:, 1], true_y_exact[:, 1]),
}

rmse_vs_truth = {}
for n in ns:
    t_net_n = data[f't_net_n{n}']
    th_n = data[f'theta_est_clt_n{n}']
    y_n = data[f'y_est_clt_n{n}']
    true_state_n, true_y_n = path_at(t_net_n)
    true_onehot_n = np.eye(n_states)[true_state_n]
    rmse_vs_truth[n] = {
        'theta': rmse(th_n, true_onehot_n),
        'Y1': rmse(y_n[:, 0], true_y_n[:, 0]),
        'Y2': rmse(y_n[:, 1], true_y_n[:, 1]),
    }

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, key, title in zip(axes, ['theta', 'Y1', 'Y2'], [r'$\theta$', r'$Y_1$', r'$Y_2$']):
    ax.plot(ns, [rmse_vs_truth[n][key] for n in ns], 'o-', color='tab:blue',
            lw=1.5, ms=4, label='ЦПТ')
    ax.axhline(rmse_exact[key], color='tab:red', ls='--', lw=1.5,
               label=f'точный Парето, $h_t={ht_exact:g}$')
    ax.set(xlabel='размер блока n', ylabel='RMSE', title=title)
    ax.set_xticks(ns)
    ax.set_xticklabels([str(n) for n in ns])
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False,
           fontsize=10, bbox_to_anchor=(.5, .99))
fig.tight_layout(rect=(0, 0, 1, .90))
plt.show()

head = f'{"фильтр":<28}{"RMSE theta":>12}{"RMSE Y1":>10}{"RMSE Y2":>10}'
print(head)
print('-' * len(head))
print(f'{"точный Парето, h_t=" + f"{ht_exact:g}":<28}{rmse_exact["theta"]:12.4f}'
      f'{rmse_exact["Y1"]:10.4f}{rmse_exact["Y2"]:10.4f}')
for n in ns:
    r = rmse_vs_truth[n]
    print(f'{clt_label(n):<28}{r["theta"]:12.4f}{r["Y1"]:10.4f}{r["Y2"]:10.4f}')

## Прямое расхождение точного и ЦПТ-фильтров

Точный фильтр прореживается до сетки каждого $n$ (`theta_est_exact[::n]`), после чего считается прямое расхождение с ЦПТ-оценкой. Для $n=1$ сетки совпадают, поэтому эта строка непосредственно показывает эффект замены исходной плотности Парето гауссовским правдоподобием при том же шаге. Тепловая карта, как и раньше, строится только для максимального $n$ — наиболее грубого разбиения.

In [ ]:
rmse_vs_exact = {}
thinned = {}
for n in ns:
    th_thin = th_exact[::n]
    y_thin = y_exact[::n]
    th_n = data[f'theta_est_clt_n{n}']
    y_n = data[f'y_est_clt_n{n}']
    m = min(th_thin.shape[0], th_n.shape[0])
    thinned[n] = (th_thin[:m], y_thin[:m], th_n[:m], y_n[:m])
    rmse_vs_exact[n] = {
        'theta': rmse(th_thin[:m], th_n[:m]),
        'Y1': rmse(y_thin[:m, 0], y_n[:m, 0]),
        'Y2': rmse(y_thin[:m, 1], y_n[:m, 1]),
    }

fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout='constrained')
for ax, key, title in zip(axes, ['theta', 'Y1', 'Y2'], [r'$\theta$', r'$Y_1$', r'$Y_2$']):
    ax.plot(ns, [rmse_vs_exact[n][key] for n in ns], 'o-', color='tab:purple')
    ax.set(xlabel='размер блока n', ylabel='RMSE относительно точного', title=title)
    ax.set_xticks(ns)
    ax.set_xticklabels([str(n) for n in ns])
plt.show()

head = f'{"n":>6}{"RMSE theta":>12}{"RMSE Y1":>10}{"RMSE Y2":>10}'
print(head)
print('-' * len(head))
for n in ns:
    r = rmse_vs_exact[n]
    print(f'{n:>6}{r["theta"]:12.4f}{r["Y1"]:10.4f}{r["Y2"]:10.4f}')

n_max = max(ns)
th_thin, y_thin, th_n, y_n = thinned[n_max]
t_net_max = data[f't_net_n{n_max}'][:th_n.shape[0]]
theta_diff = (th_n - th_thin).T
y_diff = (y_n - y_thin).T
fig, axes = plt.subplots(2, 1, figsize=(14, 5.3), layout='constrained')
for ax, difference, labels, title in [
    (axes[0], theta_diff, [fr'$\theta_{i + 1}$' for i in range(theta_diff.shape[0])],
     f'ЦПТ минус точный: вероятности состояний (n={n_max})'),
    (axes[1], y_diff, [r'$Y_1$', r'$Y_2$'],
     f'ЦПТ минус точный: непрерывное состояние (n={n_max})'),
]:
    limit = np.max(np.abs(difference)) or 1.0
    image = ax.imshow(difference, aspect='auto', cmap='RdBu_r', vmin=-limit, vmax=limit,
                      extent=[0, t_net_max[-1] / 3600, difference.shape[0] - .5, -.5])
    ax.set(yticks=np.arange(len(labels)), yticklabels=labels,
           xlabel='время, часы', title=title)
    fig.colorbar(image, ax=ax, pad=.01, label='знаковая разность')
plt.show()

print(f'Максимум |ЦПТ - точный| для theta (n={n_max}): {np.max(np.abs(theta_diff)):.4f}')
print(f'Максимум |ЦПТ - точный| для Y (n={n_max}):     {np.max(np.abs(y_diff)):.4f}')